In [1]:
import pandas as pd
import numpy as np
import os

seeds = [42,0,1,2,3]
seed = 42

dges_results = {}
original_dge = pd.read_csv('DGE_Nivo/DGE_Origin.csv')
dges_results['Origin'] = original_dge
synthetic_datasets = [f"avatarsk5_{seed}",f"avatarsk10_{seed}",f"ctgan_{seed}",f"gaussiancopula_{seed}",
                      f"synthpop_{seed}",f"tvae_{seed}"]

for synthetic_dataset in synthetic_datasets:
    try:
        dge_syn_result = pd.read_csv(f'../DGE/DGE_Nivo/Seed_{seed}/DGE_{synthetic_dataset}.csv')
        dges_results[synthetic_dataset] = dge_syn_result
    except Exception as err:
        print(f"{synthetic_dataset}: {err}")

tvae_42: [Errno 2] No such file or directory: '../DGE/DGE_Nivo/Seed_42/DGE_tvae_42.csv'


In [4]:
from DGE import extract_significant_genes
from DGE import compute_jaccard_indices
from DGE import compare_spearman_degs

orig_genes = extract_significant_genes(dges_results["Origin"], term_col="Gene", adj_p_col="P_value", threshold=0.05)
synth_genes_dict = {k: extract_significant_genes(v, term_col="Gene", adj_p_col="P_value", threshold=0.05)
                    for k, v in dges_results.items() if k != "Origin"}
jaccard_df = compute_jaccard_indices(orig_genes, synth_genes_dict)
# jaccard_df.to_csv(f"DGE_Nivo/Seed_{seed}/JaccardIndex_{seed}_NivoBenefit.csv", index=False)
jaccard_df

,Dataset,Original_count,Synthetic_count,Intersection,Union,Jaccard
0,avatarsk5_42,1349,1190,76,2463,0.030857
1,avatarsk10_42,1349,3160,141,4368,0.032280
2,ctgan_42,1349,1519,62,2806,0.022096
3,gaussiancopula_42,1349,2045,249,3145,0.079173
4,synthpop_42,1349,742,31,2060,0.015049


In [34]:
dges_results_sig = {}
for data, dges_res in dges_results.items():
    dges_res_sig = dges_res[dges_res['Gene'].isin(orig_genes)]
    dges_results_sig[data] = dges_res_sig
spearmanDF, RankScoreDF = compare_spearman_degs(
    dges_results_sig,
    origin='Origin',
    term_col="Gene",
    lfc_col="Log2FC",
    q_col="P_value")
spearmanDF.to_csv(f"DGE_Nivo/Seed_{seed}/Spearman_{seed}_NivoBenefit.csv", index=False)
RankScoreDF.to_csv(f"DGE_Nivo/Seed_{seed}/GeneRankScore_{seed}_NivoBenefit.csv", index=False)
spearmanDF

,Dataset,Spearman_rho,p_value,n_terms_used
0,avatarsk10_42,0.115173,2.231634e-05,1349
1,avatarsk5_42,0.145288,8.333154e-08,1349
2,ctgan_42,-0.029534,2.783720e-01,1349
3,gaussiancopula_42,0.343010,1.743033e-38,1347
4,synthpop_42,0.079936,3.304193e-03,1349
